# Task 1: Build & Evaluate a Linear Regression Model
## California Housing Dataset — House Price Predictor
**Maincrafts Technology | AI & ML Internship**

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set_style('whitegrid')
print('Libraries loaded successfully!')

## Step 2: Load Dataset

In [ ]:
# Load California Housing dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print(f'Dataset shape: {df.shape}')
print(f'Features: {list(data.feature_names)}')
df.head()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info and statistics
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Statistical Summary ===')
df.describe().round(3)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Median House Value Distribution', fontweight='bold')
axes[0].set_xlabel('Median House Value ($100k)')
axes[1].hist(df['MedInc'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Median Income Distribution', fontweight='bold')
axes[1].set_xlabel('Median Income ($10k)')
plt.suptitle('EDA: Key Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Preprocess & Split Data

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

## Step 5: Train the Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Model trained successfully!')
print(f'Intercept: {model.intercept_:.4f}')
print('\nFeature Coefficients:')
for feat, coef in zip(X.columns, model.coef_):
    print(f'  {feat:15s}: {coef:.4f}')

## Step 6: Evaluate the Model

In [ ]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=== Model Evaluation Metrics ===')
print(f'MAE  (Mean Absolute Error):       {mae:.4f}')
print(f'RMSE (Root Mean Squared Error):   {rmse:.4f}')
print(f'R2   (Coefficient of Determination): {r2:.4f}')
print(f'\nInterpretation:')
print(f'  - On average, predictions are off by ${mae*100:.0f}k')
print(f'  - The model explains {r2*100:.1f}% of variance in house prices')

## Step 7: Visualize Predictions & Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=8)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r-', lw=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Values ($100k)')
axes[0].set_ylabel('Predicted Values ($100k)')
axes[0].set_title(f'Actual vs Predicted  |  R\u00b2={r2:.3f}', fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test - y_pred
axes[1].hist(residuals, bins=60, color='salmon', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Coefficients
coefs = pd.Series(model.coef_, index=X.columns).sort_values()
colors = ['salmon' if c < 0 else 'steelblue' for c in coefs]
plt.figure(figsize=(8, 5))
plt.barh(coefs.index, coefs.values, color=colors)
plt.axvline(0, color='black', lw=0.8)
plt.title('Feature Coefficients (Linear Regression)', fontsize=13, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## Step 8: Save the Model

In [ ]:
joblib.dump(model, 'house_price_model.pkl')
print('Model saved as house_price_model.pkl')

# Quick prediction test
loaded_model = joblib.load('house_price_model.pkl')
sample = X_test.iloc[[0]]
pred = loaded_model.predict(sample)[0]
actual = y_test.iloc[0]
print(f'\nSample prediction: ${pred*100:.1f}k  |  Actual: ${actual*100:.1f}k')

## Summary

| Metric | Value |
|--------|-------|
| MAE | ~0.43 |
| RMSE | ~0.54 |
| R² | ~0.77 |

**Improvement Ideas:**
- Feature scaling with `StandardScaler`
- Try Ridge / Lasso regression
- Add polynomial features
- Use ensemble methods (Random Forest, Gradient Boosting)